## Import Packages and Mount Drive

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import tensorflow as tf
from tensorflow import keras

print(tf.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

.

.

.
# Load Raw Data and Extract Acceleration Signals
In the CNN practice, we use **Spectrograms** instead of the vector features (e.g., RMS, max, min).  
Here, the focus is on the **acceleration sensor signals**, which will later be converted into spectrograms (via STFT) for CNN training.

- We first load the 360 raw datasets:  
  - **Normal**: 180 samples (`Normal_1` ~ `Normal_180`)  
  - **Abnormal**: 180 samples (`Abnormal_1` ~ `Abnormal_180`)  

- From each dataset, we extract only the **second column**, which corresponds to the acceleration signal.  

- Finally, we arrange the extracted signals into a single array:  
  - Each row represents one sample (Normal or Abnormal).  
  - This array will serve as the input for later signal processing and CNN-based learning.

In [ ]:
NoOfData = 180

for i in range(NoOfData):

    temp_path1 = f'https://github.com/ljwg3000/UNT_MEEN/blob/main/AI_tutorial/Dataset/Normal_{i+1}?raw=true'
    temp_path2 = f'https://github.com/ljwg3000/UNT_MEEN/blob/main/AI_tutorial/Dataset/Abnormal_{i+1}?raw=true'

    exec(f"Normal_{i+1}   = pd.read_csv(temp_path1 , sep=',' , header=None)")
    exec(f"Abnormal_{i+1} = pd.read_csv(temp_path2 , sep=',' , header=None)")

In [ ]:
DataLength = len(Normal_1)

AccData_Nor = pd.DataFrame(np.zeros((NoOfData, DataLength)))
AccData_Abn = pd.DataFrame(np.zeros((NoOfData, DataLength)))

for i in range(NoOfData):
  exec(f"tempNormal   = Normal_{i+1}")
  exec(f"tempAbnormal = Abnormal_{i+1}")

  AccData_Nor.iloc[i,:] = tempNormal.iloc[:,1]
  AccData_Abn.iloc[i,:] = tempAbnormal.iloc[:,1]

AccData = np.array(pd.concat([AccData_Nor, AccData_Abn], axis=0))
AccData.shape

# Convert Acceleration Data into Spectrogram (STFT)

To use CNNs, we need **2D input data** (matrices or images).  
Instead of using simple statistical features (RMS, max, min, etc.), we transform the raw acceleration signals into **time–frequency spectrograms** using the **Short-Time Fourier Transform (STFT)**.

- **STFT** splits the signal into small overlapping windows and applies the Fourier Transform on each window.  
- This produces a **spectrogram**, which shows how the frequency content of the signal changes over time.  
- Each spectrogram is a 2D matrix (time × frequency), making it suitable as input for CNN models.  

Parameters:  
- `nperseg`: number of samples in each segment (controls frequency resolution).  
- `noverlap`: number of overlapping samples (controls time resolution).  

By adjusting these values, we can change the **resolution of the spectrogram**.

In [ ]:
from scipy import signal

Fs = 12800  # Sampling Frequency
f,t,AccSTFT = signal.spectrogram(AccData, Fs, nperseg = 78, noverlap = 10)
AccSTFT.shape

We can visually compare the spectrograms of **Normal** and **Abnormal** signals to observe differences in frequency patterns.

In [ ]:
idx = 1  # Select index (1~180)

plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.pcolormesh(t, f, AccSTFT[idx-1], cmap='jet')
plt.title(f"STFT (Normal_{idx})", fontsize=15)
plt.xlabel('Time(s)', fontsize=12)
plt.ylabel('Frequency(Hz)', fontsize=12)
plt.colorbar()

plt.subplot(1,2,2)
plt.pcolormesh(t, f, AccSTFT[idx+NoOfData-1], cmap='jet')
plt.title(f"STFT (Abnormal_{idx})", fontsize=15)
plt.xlabel('Time(s)', fontsize=12)
plt.colorbar()

plt.show()

### Reshaping Spectrograms and Adding Channel Information

The extracted spectrograms have a shape of **(40 × 40)** for each sample.  
However, CNN models expect image-like input data with an additional **channel dimension**.  

- In this case, each spectrogram has only **one value per pixel** (like a grayscale image).  
  → Therefore, we add a channel dimension of **1**, making the shape `(40, 40, 1)`.  

- If we had **RGB images**, each pixel would contain 3 values (Red, Green, Blue).  
  → The input shape would then be `(40, 40, 3)`.  

- Similarly, if we used spectrograms from **3 different sensors** (e.g., acceleration, voltage, and current),  
  each location in the spectrogram would have 3 values.  
  → The input shape would again be `(40, 40, 3)`.  

By adding this channel dimension, we make the data compatible with CNN architectures, which are designed to process multi-channel images.


In [ ]:
NormalSet   = AccSTFT[:NoOfData]
AbnormalSet = AccSTFT[NoOfData:]

NoOfSensor  = 1
NormalSet   = NormalSet.reshape(NormalSet.shape[0], NormalSet.shape[1], NormalSet.shape[2], NoOfSensor)
AbnormalSet = AbnormalSet.reshape(AbnormalSet.shape[0], AbnormalSet.shape[1], AbnormalSet.shape[2], NoOfSensor)

NormalSet.shape, AbnormalSet.shape

.

.

.

.

## Split Training & Test Data
- Use 'train_test_split' function
- It randomly samples the training and testing data according to the designated ratio.

In [ ]:
from sklearn.model_selection    import train_test_split

# Designate test data ratio
TestData_Ratio = 0.2

TrainData_Nor, TestData_Nor = train_test_split(NormalSet  , test_size=TestData_Ratio, random_state=777)
TrainData_Abn, TestData_Abn = train_test_split(AbnormalSet, test_size=TestData_Ratio, random_state=777)

print(TrainData_Nor.shape, TestData_Nor.shape)
print(TrainData_Abn.shape, TestData_Abn.shape)

## Data Labling (One-hot Encoding)
- `[1, 0]` → Normal  
- `[0, 1]` → Abnormal  

In [ ]:
TrainLabel_Nor = np.zeros((TrainData_Nor.shape[0],2))
TrainLabel_Abn = np.ones( (TrainData_Abn.shape[0],2))
TestLabel_Nor  = np.zeros((TestData_Nor.shape[0],2))
TestLabel_Abn  = np.ones( (TestData_Abn.shape[0],2))

TrainLabel_Nor[:,0] = 1  # [1,0]: Normal
TrainLabel_Abn[:,0] = 0  # [0,1]: Abnormal
TestLabel_Nor[:,0]  = 1  # [1,0]: Normal
TestLabel_Abn[:,0]  = 0  # [0,1]: Abnormal

print(TrainLabel_Nor.shape, TestLabel_Nor.shape)
print(TrainLabel_Abn.shape, TestLabel_Abn.shape)

## Finalizing Data and Label Preparation

In [ ]:
TrainData  = np.concatenate([TrainData_Nor , TrainData_Abn ], axis=0)
TestData   = np.concatenate([TestData_Nor  , TestData_Abn  ], axis=0)
TrainLabel = np.concatenate([TrainLabel_Nor, TrainLabel_Abn], axis=0)
TestLabel  = np.concatenate([TestLabel_Nor , TestLabel_Abn ], axis=0)

print(TrainData.shape,  TestData.shape)
print(TrainLabel.shape, TestLabel.shape)

.

.

.


## Setting hyperparameters for training CNN (Convolutional Neural Network)

In [ ]:
learningRate  = 0.0001
Epoch         = 1000

## Designing an CNN architecture (based on Keras)

### Key Design Factors in CNN Architecture

In Convolutional Neural Networks (CNNs), several hyperparameters control how feature extraction is performed from the input spectrograms:

- **Filters (kernels)**:  
  The number of filters determines how many different feature maps are learned.  
  Each filter detects a specific type of pattern (e.g., edges, frequency components).  
  - Example: `filters=2` in the first Conv2D layer creates 2 feature maps.

- **Kernel size**:  
  Defines the size of the sliding window (e.g., `3×3`).  
  Larger kernels capture broader patterns, while smaller kernels focus on local details.

- **Stride**:  
  Determines how far the filter moves at each step.  
  - `strides=(1,1)` means the filter shifts by 1 pixel at a time (fine-grained scanning).  
  Larger strides reduce the output size but may skip details.

- **Padding**:  
  Controls how borders of the input are handled.  
  - `'same'` padding preserves the input size by filling edges with zeros.  
  - `'valid'` padding does not add padding, resulting in smaller output.

- **Activation function (ReLU)**:  
  - Applied after each convolution operation.  
  - ReLU (`f(x) = max(0, x)`) removes negative values and introduces non-linearity,  
  - allowing the network to learn complex and non-linear patterns in the data.


- **Pooling layers**:  
  Reduce the size of the feature maps while keeping important patterns.  
  - `pool_size=(2,2)` halves the dimensions by taking the maximum value in each 2×2 region.  
  Pooling helps make the model more efficient and reduces overfitting.

By combining these elements, CNNs progressively extract hierarchical features from spectrograms — starting from simple

- Types of Convolution layer: https://keras.io/api/layers/convolution_layers/

- Types of Pooling layer: https://keras.io/api/layers/pooling_layers/

- Flatten layer: https://keras.io/api/layers/reshaping_layers/flatten/

In [ ]:
def CNN_model(input_data):
    keras.backend.clear_session()

    model = keras.Sequential()
    model.add(keras.layers.InputLayer(input_shape=(input_data.shape[1],input_data.shape[2],input_data.shape[3])))       # Input layer

    model.add(keras.layers.Conv2D(filters = 2, kernel_size=(3,3), strides=(1,1), padding='same', activation='relu'))    # Convolution layer 1
    model.add(keras.layers.MaxPooling2D(pool_size = (2,2), strides=(2,2)))                                              # Pooling layer 1
    model.add(keras.layers.Conv2D(filters = 4, kernel_size=(3,3), strides=(1,1), padding='same', activation='relu'))    # Convolution layer 2
    model.add(keras.layers.MaxPooling2D(pool_size = (2,2), strides=(2,2)))                                              # Pooling layer 2

    model.add(keras.layers.Flatten())                                                                                   # Flatten layer
    model.add(keras.layers.Dense(units = 10, activation='relu'))                                                        # Dense layer

    model.add(keras.layers.Dense(units = 2, activation='softmax'))                                                      # Output Layer

    model.compile(optimizer= keras.optimizers.Adam(learning_rate = learningRate),
                  loss=keras.losses.CategoricalCrossentropy(),
                  metrics=['accuracy'])
    return model

In [ ]:
# Check the model architecture and the number of parameters
CnnModel = CNN_model(TrainData)
CnnModel.summary()

## CNN Model Training

In [ ]:
tf.random.set_seed(777) # Not necessarily required

# Model traning and validation
TraingHistory  = CnnModel.fit(TrainData, TrainLabel, epochs=Epoch, verbose = 0)

In [ ]:
# Evaluation result for test data (not trained)
Loss, Accuracy = CnnModel.evaluate(TestData,  TestLabel, verbose=0)
Loss, Accuracy # The closer the Loss is to 0 and the closer the accuracy is to 1 (100%), the better.

In [ ]:
# Check the training process (Loss, Accuracy)

fig, loss_ax = plt.subplots(figsize=(8,6))
acc_ax = loss_ax.twinx()

loss_ax.plot(TraingHistory.history['loss'], label='train loss', c = 'tab:red')
loss_ax.set_xlabel('epoch', fontsize=15)
loss_ax.set_ylabel('loss', fontsize=15)
loss_ax.legend(loc='center left', fontsize=12)

acc_ax.plot(TraingHistory.history['accuracy'], label='train acc', c = 'tab:blue')
acc_ax.set_ylabel('accuracy', fontsize=15)
acc_ax.legend(loc='center right', fontsize=12)

plt.show()

Save ML model (CNN) as a file

In [ ]:
CnnModel.save('/content/drive/MyDrive/Colab Notebooks/SavedFiles/ML_Models/CNN_model.keras')

Load the saved ML model (CNN) and test

In [ ]:
LoadedModel = keras.models.load_model('/content/drive/MyDrive/Colab Notebooks/SavedFiles/ML_Models/CNN_model.keras')

Loss, Accuracy = LoadedModel.evaluate(TestData, TestLabel, verbose=0)
print('[Performance of CNN model] \n')
print('Accuracy : {:.2f}%'.format(Accuracy*100))

In [ ]:
# Predicted result
Predicted = LoadedModel.predict(TestData)

# Convert TestLabel and Predicted into vectors to calculate the confusion matrix and evaluation metrics
TestLabel_rev = np.argmax(TestLabel, axis=1)
Predicted_rev = np.argmax(Predicted, axis=1)

# Plot the confusion matrix
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Calculate the confusion matrix
cm = confusion_matrix(TestLabel_rev, Predicted_rev)

plt.figure(figsize=(6, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap=plt.cm.Blues, cbar=False, square=True)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix of the CNN Model")
plt.show()

from sklearn import metrics

# Calculate the evaluation metrics
accuracy  = metrics.accuracy_score(TestLabel_rev, Predicted_rev)
precision = metrics.precision_score(TestLabel_rev, Predicted_rev)
recall    = metrics.recall_score(TestLabel_rev, Predicted_rev)
f1_score  = metrics.f1_score(TestLabel_rev, Predicted_rev)

# Print the evaluation metrics
print("\n\n")
print(f"CNN Model Evaluation:\n")
print(f"Accuracy : {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall   : {recall:.2f}")
print(f"F1 Score : {f1_score:.2f}")